# RF-DETR — Appearance ReID tracking comparison (Colab)

Compare person tracking with appearance **ReID off vs on** over one clip.

The **same detections** are fed through two tracking pipelines that differ only in `reid_enabled`, so any change in track-id stability is attributable to ReID alone. The lightweight ReID is a CPU-friendly HSV torso-color histogram (no embedding model, no extra dependencies).

**How to read the results:** if `mean active` (average people per frame) stays about the same while `unique track ids` drops, ReID is successfully reviving ids for people who left and returned. If the average count itself drops, ids are being wrongly merged — raise `--reid-similarity`.

> Tip: `Runtime → Change runtime type → GPU` before running (keypoint inference is slow on CPU).

In [ ]:
!nvidia-smi -L || echo 'No GPU detected — inference will be slow. Runtime > Change runtime type > GPU.'

## 1. Clone and install

In [ ]:
import os

REPO_URL = 'https://github.com/shingo257/rf-detr.git'
BRANCH = 'develop'

if not os.path.exists('rf-detr'):
    !git clone --branch {BRANCH} --depth 1 {REPO_URL}
%cd rf-detr
!pip -q install -e .
print('\nInstalled. If imports fail below, use Runtime > Restart session, then re-run from this cell (the clone is skipped automatically).')

## 2. Choose a video

Defaults to a public *people-walking* clip (people occlude each other — good for ReID). To use your own footage, uncomment the upload lines. The bundled `sample/*.mov` files are private and are **not** in the repo, so they are not available here.

In [ ]:
import os
import cv2

# Default: a public people-walking clip (people occlude each other — good for ReID).
VIDEO_URL = 'https://media.roboflow.com/supervision/video-examples/people-walking.mp4'
VIDEO = 'demo.mp4'
if not os.path.exists(VIDEO):
    !wget -q -O {VIDEO} {VIDEO_URL}

# --- To use your own video instead, uncomment (overrides the download above): ---
# from google.colab import files
# uploaded = files.upload()
# VIDEO = next(iter(uploaded))

cap = cv2.VideoCapture(VIDEO)
assert cap.isOpened(), f'Could not open {VIDEO} — re-run this cell or upload a valid video.'
print(f'Using {VIDEO}: {int(cap.get(cv2.CAP_PROP_FRAME_COUNT))} frames, '
      f'{int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))}x{int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}')
cap.release()

## 3. Run the comparison (ReID off vs on)

The first run downloads the keypoint model weights. `--max-frames` keeps it quick; raise or drop it for your clip.

In [ ]:
!rfdetr-demo compare-reid --source {VIDEO} --max-frames 600 --json reid_metrics.json

import json
print('\n' + json.dumps(json.load(open('reid_metrics.json')), indent=2, ensure_ascii=False))

## 4. Sweep the thresholds

Vary both the cost-blend `--reid-weight` and the revival `--reid-similarity`. `weight=0.0` isolates gallery revival (no per-frame appearance blend). Pick the **highest** `similarity` that still consolidates ids (`ids_on` well below `ids_off`) **while** `mean_on` stays close to `mean_off` — that is the setting most resistant to merging different people. If `ids_on` stays flat across low thresholds, the descriptor separates cleanly and the useful knob is `weight`, not `similarity`.

In [ ]:
import json, subprocess
import pandas as pd

rows = []
for weight in [0.0, 0.3, 0.6]:            # 0.0 = gallery revival only (no cost blend)
    for similarity in [0.5, 0.7, 0.8, 0.9, 0.95]:
        subprocess.run(
            ['rfdetr-demo', 'compare-reid', '--source', VIDEO, '--max-frames', '600',
             '--reid-weight', str(weight), '--reid-similarity', str(similarity), '--json', 'sweep.json'],
            check=True,
        )
        data = json.load(open('sweep.json'))
        off, on = data['reid_off'], data['reid_on']
        rows.append({
            'weight': weight,
            'similarity': similarity,
            'ids_off': off['unique_ids'],
            'ids_on': on['unique_ids'],
            'mean_off': round(off['mean_active'], 2),
            'mean_on': round(on['mean_active'], 2),
            'std_on': round(on['count_std'], 2),
        })

pd.DataFrame(rows)   # pick the highest similarity that still cuts ids while mean_on stays ~ mean_off

## 5. Visual check — id-labeled OFF vs ON

Render both videos from the **same detections** with track ids drawn (the number by each box; a trailing `*` means the track is being held through an occlusion). Set `--reid-similarity` to the value you picked from the sweep.

**What to look for:**
- ✅ Good: in **ON**, a person keeps the same id/color across a brief occlusion, where **OFF** flips them to a new id.
- ⚠️ Over-merge: in **ON**, two clearly different people share one id/color → raise `--reid-similarity`.

In [ ]:
# Write id-labeled OFF and ON videos from one run (same detections), then play both inline.
!rfdetr-demo compare-reid --source {VIDEO} --max-frames 600 --reid-similarity 0.85 --write-video --out-dir compare_out

from base64 import b64encode
from IPython.display import HTML, display

def show(path, label):
    mp4 = b64encode(open(path, 'rb').read()).decode()
    display(HTML(
        f'<p><b>{label}</b> — number = track id, <code>*</code> = held during occlusion</p>'
        f'<video width=640 controls><source src="data:video/mp4;base64,{mp4}" type="video/mp4"></video>'
    ))

show('compare_out/reid_off.mp4', 'ReID OFF')
show('compare_out/reid_on.mp4', 'ReID ON (similarity 0.85)')

## Tuning cheatsheet

All default off; enable per run via env vars or the `compare-reid` flags.

| Env var | Flag | Meaning | Default |
| --- | --- | --- | --- |
| `RFDETR_TRACK_REID` | (compare-reid always runs both) | Enable appearance ReID | off |
| `RFDETR_REID_WEIGHT` | `--reid-weight` | Appearance vs IoU cost blend (0..1) | 0.3 |
| `RFDETR_REID_SIMILARITY` | `--reid-similarity` | Min histogram match to revive an id | 0.5 |
| `RFDETR_REID_GALLERY_FRAMES` | `--reid-gallery-frames` | How long a retired id stays revivable | 60 |
| `RFDETR_REID_EMA` | — | Descriptor smoothing (0..1) | 0.9 |

If the color histogram proves too weak for your footage (e.g. everyone wears similar colors), the descriptor in `src/rfdetr_demo/tracking/appearance.py` can later be swapped for a small ONNX/OSNet embedding behind the same interface.